In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "0,1,2,3"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader

/mnt/petrelfs/zhangshilin/anaconda3/envs/deepscaler/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-24 10:19:19,385	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
model_path = "/mnt/petrelfs/share_data/huzican/Qwen2.5-7B-orz-tok"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).cuda()

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.46it/s]


In [3]:
train_data_path = "dataset/valid.all.parquet"
train_dataset = RLHFDataset(parquet_files=train_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

train_dataloader = DataLoader(dataset=train_dataset,
                            batch_size=2,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 6023
filter dataset len: 6021


In [ ]:
for test_data in train_dataloader:
    seq = tokenizer.batch_decode(test_data['input_ids'],skip_special_tokens=True)
    input_ids = test_data['input_ids'].to(model.device)
    

In [4]:
for test_data in train_dataloader:
    print(test_data.keys())
    seq = tokenizer.batch_decode(test_data['input_ids'],skip_special_tokens=True)
    print(seq)
    # 准备输入
    input_ids = test_data['input_ids'].to(model.device)
    print(input_ids.shape)
    attention_mask = test_data['attention_mask'].to(model.device)
    group_rollout = []
    for i in range(8):
        # 生成文本
        with torch.no_grad():
            gene = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=8192,  # 生成新token的数量
                do_sample=True,
                temperature=1.0,
                top_p=1.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                # repetition_penalty=1.2  # 避免重复
            )
        
        # 解码生成的序列
        original_length = input_ids.shape[1]
        print(original_length)
        new_tokens = gene[:, original_length:]
        generated_texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        group_rollout.append(generated_texts)
    
    for i in range(len(group_rollout)):
        print(f"********{i}**********")
        print(group_rollout[i])
        print("******************")
    
    break


dict_keys(['input_ids', 'attention_mask', 'position_ids', 'data_source', 'ability', 'reward_model', 'extra_info', 'index'])
['Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: “<think>\n {thoughts} </think>\n”. Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After “</think>\n,” in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \\boxed{} for closed-form results like multiple choices or mathematical solutions. User: This is the problem:\nIn a table tennis tournament every participant played every other participant e

In [5]:
import re

def extract_formulas(response):
    # 定义不同格式公式的正则模式
    patterns = [
        r'\\\[([^\]]*?)\\\]',     # \[ \]
        r'\\\(([^\)]*?)\\\)',     # \( \)
        r'\$([^\$]*?)\$'          # $ $
    ]
    
    # 存储所有找到的公式
    formulas = set()
    
    # 对每个模式进行匹配
    for pattern in patterns:
        matches = re.findall(pattern, response)
        # 将找到的公式添加到集合中（自动去重）
        formulas.update(matches)
    
    return list(formulas)

# text = ['To solve this problem, we need to determine the areas of circles A and B, and then find the ratio of these areas. The key here is to understand that for any right triangle inscribed in a circle, the hypotenuse of the triangle is the diameter of the circle.\n\n### Step-by-Step Analysis\n\n1. **Determine the diameter of each circle:**\n\n   - For the $3-4-5$ right triangle in circle $A$: The hypotenuse (which is the diameter of the circle) is $5$. Therefore, the radius $r_A$ of circle $A$ is $\\frac{5}{2}$.\n   - For the $5-12-13$ right triangle in circle $B$: The hypotenuse (which is the diameter of the circle) is $13$. Therefore, the radius $r_B$ of circle $B$ is $\\frac{13}{2}$.\n\n2. **Calculate the areas of the circles:**\n   - Area of circle $A$ ($A_A$) = $\\pi \\left( \\frac{5}{2} \\right)^2 = \\frac{25\\pi}{4}$.\n   - Area of circle $B$ ($A_B$) = $\\pi \\left( \\frac{13}{2} \\right)^2 = \\frac{169\\pi}{4}$.\n\n3. **Find the ratio of the areas of circle $A$ to circle $B$:**\n   - Ratio = $\\frac{A_A}{A_B} = \\frac{\\frac{25\\pi}{4}}{\\frac{169\\pi}{4}} = \\frac{25}{169}$.\n\n4. **Simplify the ratio and find $m+n$:**\n   - The ratio $\\frac{25}{169}$ is already in its simplest form since 25 and 169 are relatively prime.\n   - Thus, $m=25$ and $n=169$, therefore $m+n = 25 + 169 = 194$.\n\n</think>\n\nSolution: The ratio of the area of circle $A$ to the area of circle $B$ is $\\frac{25}{169}$. Therefore, $m+n = \\boxed{194}$.', "We start by considering the expression \\(| | x | - 1 | + | | y | - 1 | \\le 1\\) and interpret it geometrically. The given inequality represents a region in the coordinate plane.\n\nTo dissect this, we observe that \\(|x|\\) and \\(|y|\\) are absolute values, which means \\(|x|\\) is the same as \\(x\\) if \\(x \\geq 0\\) and \\(|x|\\) is the same as \\(-x\\) if \\(x < 0\\). However, since these absolute values are then further subjected to another absolute value (i.e., \\(|-|x|-1|\\) and \\(|-|y|-1|\\)), we will consider the range \\(|x| \\geq 0\\) and \\(|y| \\geq 0\\) and then extend the result to all quadrants.\n\nLet's consider \\(|x| \\geq 0\\) and \\(|y| \\geq 0\\) first:\n\n1. \\(|x| - 1\\) can be either \\(x - 1\\) or \\(-x - 1\\).\n2. \\(|y| - 1\\) can be either \\(y - 1\\) or \\(-y - 1\\).\n\nThis means we need to consider four quadrants within the first quadrant, and by symmetry, the same will extend to the other three quadrants. This results in an analysis of the following equations:\n\n\\( |x| - 1 + |y| - 1 \\le 1 \\)\nwhich simplifies to \n\\( |x| + |y| - 2 \\le 1 \\) or \n\\( |x| + |y| \\le 3 \\).\n\nThis equation defines a square centered at (1, 1) with a side length of 2sqrt(2) (as the origin is at the center, half of the distance to any vertex is sqrt(2), thus the full side is 2*sqrt(2)).\n\nTherefore, the region described in the problem is a square with side 2sqrt(2).\n\n</think>\n\nSolution: The area of the square is \\((2\\sqrt{2})^2 = 8\\). And because we've considered all four quadrants, we multiply by 4 to get the area of the full region. Therefore, the area of the region is \\(4 \\times 8 = \\boxed{32}\\)."]
# extract_formulas(text[0])

formulas = []
for i in range(len(group_rollout)):
    formulas.append(extract_formulas(group_rollout[i][0]))

for i in range(len(formulas)):
    print(f'response {i}:')
    print(formulas[i])

response 0:
[' R = 2 \\times 3 = 6 ', ' W_L = 1.4 W_R ', '\n   \\frac{3L(3L-1)}{2} = \\frac{3 \\times 3 \\times 2}{2} = \\frac{18}{2} = 9\n   ', ' \\frac{12}{2} ', '\n   x + 1.4x = 2.4x = 36 \\Rightarrow x = \\frac{36}{2.4} = 15\n   ', ' W_R ', ' R = 6 ', ' W_L = 1.4 \\times 15 = 21 ', ' L = 2 ', ' 15 + 21 = 36 ', ' \\frac{9 \\times 8}{2} = 36 ', '\n   2.4 W_R = \\frac{3L(3L-1)}{2}\n   ', ' n ', ' L = 3 ', ' R = 2L ', '\n   W_R + W_L = W_R + 1.4 W_R = 2.4 W_R\n   ', ' \\boxed{27} ', ' L ', ' R ', ' W_R = x ', ' W_L ', ' W_R = 15 ', ' L + R = 3L ', ' W_L = 1.4x ', '\n   W_R = \\frac{3L(3L-1)}{4.8}\n   ', ' L = 1 ']
response 1:
[' 1.4 \\times W_R + W_R = \\text{Total Games} ', ' W_L + W_R = \\text{Total Games} ', 'L=20', 'W_L = 1.4 \\times W_R', ' W_R = \\frac{\\text{Total Games}}{2.4} ', ' \\text{Total Games} = \\frac{90 \\times 89}{2} = 4005 ', 'P = L + R', ' W_L = 1.4 \\times W_R ', ' \\text{Total Games} = \\frac{30 \\times 29}{2} = 435 ', ' W_R = \\frac{435}{2.4} = 181.25 ', 'L=30', 

In [6]:
all_formulas = sum(formulas, [])
print(len(set(all_formulas)))

118


In [7]:
def calculate_unique_diversity(formulas, current_index):
    if not formulas[current_index]:  # 如果当前response为空
        return 0
    
    # 获取所有其他response中的公式
    other_formulas = set()
    for i in range(len(formulas)):
        if i != current_index:
            other_formulas.update(formulas[i])
    
    # 获取当前response中的公式
    current_formulas = set(formulas[current_index])
    
    # 计算只在当前response中出现的公式（独特公式）
    unique_formulas = current_formulas - other_formulas
    
    # 计算多样性指标 D_eq
    D_eq = len(unique_formulas) / len(formulas[current_index])

    print(f'{len(unique_formulas)} / {len(formulas[current_index])}')
    
    return D_eq

# 对每个response计算多样性指标
for i in range(len(formulas)):
    diversity = calculate_unique_diversity(formulas, i)
    print(f'response {i} diversity: {diversity:.3f}')

16 / 26
response 0 diversity: 0.615
37 / 38
response 1 diversity: 0.974
11 / 19
response 2 diversity: 0.579
13 / 18
response 3 diversity: 0.722
response 4 diversity: 0.000
11 / 21
response 5 diversity: 0.524
17 / 20
response 6 diversity: 0.850
response 7 diversity: 0.000


In [9]:
diversity = []
for i in range(len(formulas)):
    diversity.append(calculate_unique_diversity(formulas, i))

print(diversity)

16 / 26
37 / 38
11 / 19
13 / 18
11 / 21
17 / 20
[0.6153846153846154, 0.9736842105263158, 0.5789473684210527, 0.7222222222222222, 0, 0.5238095238095238, 0.85, 0]


In [12]:
import heapq
indices = heapq.nlargest(4, range(len(diversity)), key=lambda i: diversity[i])
indices

[1, 6, 3, 0]